# Fixture amendment — post-migration dev (contract rev 10, 2026-07-28)

Run this **in the same kernel as `fetch_fixture_json_v2.ipynb`** (needs `auth_manager`, `BASE_URL`, `VIEW_DETAILS_PATH`, `SERVER_ID`, `OUT_DIR`, `sanitize`, and `SOURCE_ENV` already defined).

What it does, per contract rev 10:
1. Overwrites `fixtures/selection.json` with the amended **8-fixture** set
   (`plain` re-selected as `inventory_daily_balance_fact`; `error_500` marked synthetic; `second_vdb` added).
2. Fetches the two fixtures that changed or are new: `plain`, `second_vdb`.
3. Generates the **synthetic** `error_500.json` (shape-identical to a real error record, `_synthetic: true`).
4. Sanity-prints redaction checks on `custom_tab.json`.

The 4 already-good fixtures from the earlier run (`doc_url`, `access_role`, `ods_link`, `resource_rich`) and `custom_tab` are left as-is.

In [12]:
import html
import json
import os
import re
import time

import requests
import truststore
import webview
from requests_oauthlib import OAuth2Session

truststore.inject_into_ssl()  # handles LANL internal TLS certs

required = ["AUTH_FLOW_CLIENT_ID", "AUTH_FLOW_CLIENT_SECRET",
            "REDIRECT_URI", "AUTH_URL", "TOKEN_URL", "SCOPE"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}")
print("All 6 env vars present \u2713")


class OAuthManager:
    """Complete OAuth manager that handles initial auth AND refresh"""

    def __init__(self, client_id, client_secret, redirect_uri, auth_url, token_url, scope):
        self.client_id = client_id
        self.client_secret = client_secret
        self.redirect_uri = redirect_uri
        self.auth_url = auth_url
        self.token_url = token_url
        self.scope = scope
        self.oauth = None
        self.token = None
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None

    def authenticate(self):
        self.oauth = OAuth2Session(self.client_id, redirect_uri=self.redirect_uri, scope=self.scope)
        authorization_url, state = self.oauth.authorization_url(self.auth_url)
        print("Authenticating...")
        print(f"[DEBUG] authorization_url = {authorization_url}")
        authorization_response = self._launch_browser_auth(authorization_url)
        if not authorization_response:
            print("\u2717 Authentication failed")
            return False
        self.token = self.oauth.fetch_token(
            self.token_url, authorization_response=authorization_response,
            client_secret=self.client_secret,
        )
        self.access_token = self.token["access_token"]
        self.refresh_token = self.token.get("refresh_token")
        self.token_expiry = time.time() + self.token.get("expires_in", 3600)
        print("\u2713 Authentication successful")
        return True

    def _launch_browser_auth(self, authorization_url):
        authorization_response = None

        def on_loaded():
            nonlocal authorization_response
            current_url = window.get_current_url()
            print(f"[DEBUG] page loaded: {current_url}")
            if current_url and self.redirect_uri in current_url:
                authorization_response = current_url
                print("\u2713 Captured Authorization")
                window.hide()
                time.sleep(2.5)
                window.destroy()

        window = webview.create_window(
            "OAuth Authorization", authorization_url, width=800, height=600,
            resizable=True, on_top=True,
        )
        window.events.loaded += on_loaded
        webview.start(private_mode=True)  # never reuse a cached LANL SSO session
        return authorization_response

    def _is_token_expired(self):
        if not self.token_expiry:
            return True
        return time.time() >= (self.token_expiry - 60)

    def _refresh_access_token(self):
        if not self.refresh_token:
            print("\u26a0 No refresh token available, re-authenticating...")
            return self.authenticate()
        try:
            new_token = self.oauth.refresh_token(
                self.token_url, refresh_token=self.refresh_token,
                client_id=self.client_id, client_secret=self.client_secret,
            )
            self.token = new_token
            self.access_token = new_token["access_token"]
            self.refresh_token = new_token.get("refresh_token", self.refresh_token)
            self.token_expiry = time.time() + new_token.get("expires_in", 3600)
            print("\u2713 Token refreshed successfully")
            return True
        except Exception as e:
            print(f"\u2717 Refresh failed: {e}")
            print("Re-authenticating...")
            return self.authenticate()

    def _ensure_authenticated(self):
        if not self.access_token or self._is_token_expired():
            if self.refresh_token:
                self._refresh_access_token()
            else:
                self.authenticate()

    def get(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.get(url, headers=headers, **kwargs)

    def post(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.post(url, headers=headers, **kwargs)


# ---------------- PROD configuration ----------------
BASE_URL = "https://datacatalog.lanl.gov/denodo-data-catalog"  # PRODUCTION
TARGET_DB = "dataportal"
SERVER_ID = 1            # confirm unchanged on PROD post-migration
ELEMENT_TYPE = "VIEWS"   # confirmed via Swagger on 2026-07-27
MAX_VIEWS_PER_GROUP = 3  # widened sample vs. the -b run

auth_manager = OAuthManager(
    client_id=os.getenv("AUTH_FLOW_CLIENT_ID"),
    client_secret=os.getenv("AUTH_FLOW_CLIENT_SECRET"),
    redirect_uri=os.getenv("REDIRECT_URI"),
    auth_url=os.getenv("AUTH_URL"),
    token_url=os.getenv("TOKEN_URL"),
    scope=os.getenv("SCOPE"),
)
auth_manager.authenticate()


All 6 env vars present ✓
Authenticating...
[DEBUG] authorization_url = https://idp.lanl.gov/as/authorization.oauth2?response_type=code&client_id=REDACTED&redirect_uri=https%3A%2F%2Fdatacatalog-d.lanl.gov%2Foauth%2F2.0%2FredirectURL.jsp&scope=den-datacat-adm&state=REDACTED
[DEBUG] page loaded: https://weblogin.lanl.gov/login
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
✓ Authentication successful


True

In [9]:
import html
import re

# --- Config (same as fetch_fixture_json_v2) ---
BASE_URL = "https://datacatalog-d.lanl.gov/denodo-data-catalog"
VIEW_DETAILS_PATH = "/public/api/view-details"
SERVER_ID = 1
SOURCE_ENV = "dev"
OUT_DIR = "fixtures"

# --- Sanitizer (same as fetch_fixture_json_v2) ---
def sanitize(obj):
    """Redact person-identifying strings; keep technical content intact."""
    if isinstance(obj, dict):
        return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize(v) for v in obj]
    if isinstance(obj, str):
        s = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "REDACTED_EMAIL", obj)
        s = re.sub(r"https?://pbplus\.lanl\.gov[^\s\"'<>]*", "REDACTED_PERSON_URL", s)
        s = re.sub(r"([?&;]user=)[^&;\"'<>\s]+", r"\1REDACTED_UID", s, flags=re.IGNORECASE)
        s = re.sub(r"(UID=)[^;\"'<>\s]+", r"\1REDACTED_UID", s, flags=re.IGNORECASE)
        return s
    return obj

print("Config + sanitize loaded ✓  (OUT_DIR =", OUT_DIR + ")")

Config + sanitize loaded ✓  (OUT_DIR = fixtures)


In [10]:
import json, os, time

AMENDED = {
    "plain": {
        "view_name": "inventory_daily_balance_fact",
        "db": "dataportal",
        "selection_method": ("re-selected 2026-07-28: original 'access_area_type' returns HTTP 404 on "
                             "post-migration dev (absent from current list); re-selection intersected "
                             "probe zero-signal views with the live list and verified HTTP 200"),
        "hits": [], "n_signals": 0,
    },
    "error_500": {
        "view_name": "vlanlchangerequesttask",
        "db": "dataportal",
        "selection_method": ("SYNTHETIC as of 2026-07-28: the pre-migration 12-view HTTP-500 cohort no "
                             "longer reproduces (post-migration dev lists 30 prod-origin vlanl*task "
                             "views, all 200). Fixture is generated until a live failure is observed "
                             "again (contract rev 10 b)."),
        "synthetic": True, "hits": [], "n_signals": 0,
    },
    "second_vdb": {
        "view_name": "i_ods_pa_cpnt_evthst",
        "db": "ops_core_publication",
        "selection_method": ("manual 2026-07-28: first empirically observed second VDB on post-migration "
                             "dev (4 views, all HTTP 200); proves non-dataportal db_name handling "
                             "(contract rev 10 c/d). Not in probe_results.db."),
        "hits": [], "n_signals": 0,
    },
}

SEL_PATH = os.path.join(OUT_DIR, "selection.json")
with open(SEL_PATH) as f:
    selection = json.load(f)
selection.update(AMENDED)
with open(SEL_PATH, "w") as f:
    json.dump(selection, f, indent=2)
print(f"selection.json now has {len(selection)} fixtures:", ", ".join(selection))

selection.json now has 8 fixtures: plain, doc_url, access_role, ods_link, error_500, resource_rich, custom_tab, second_vdb


## Fetch the changed/new fixtures (`plain`, `second_vdb`)

In [13]:
fetched_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
for label in ["plain", "second_vdb"]:
    sel = selection[label]
    view, db = sel["view_name"], sel["db"]
    resp = auth_manager.get(
        BASE_URL + VIEW_DETAILS_PATH,
        params={"viewName": view, "databaseName": db, "serverId": SERVER_ID},
    )
    record = {
        "_fixture_label": label,
        "_view_name": view,
        "_db_name": db,
        "_http_status": resp.status_code,
        "_fetched_at": fetched_at,
        "_source_env": SOURCE_ENV,
        "_content_note": "post-migration dev (prod-equivalent content; contract rev 10 a/e)",
    }
    if resp.status_code == 200:
        record["response"] = sanitize(resp.json())
    else:
        record["response"] = None
        record["_error_body"] = resp.text[:500]
    path = os.path.join(OUT_DIR, f"{label}.json")
    with open(path, "w") as f:
        json.dump(record, f, indent=2)
    print(f"[{label}] {db}.{view} -> HTTP {resp.status_code} -> {path}")
    assert resp.status_code == 200, f"{label} did not return 200 -- investigate before proceeding"
print("Both fetched OK.")

[plain] dataportal.inventory_daily_balance_fact -> HTTP 200 -> fixtures\plain.json
[second_vdb] ops_core_publication.i_ods_pa_cpnt_evthst -> HTTP 200 -> fixtures\second_vdb.json
Both fetched OK.


## Generate the synthetic `error_500` fixture

Shape-identical to a real error record produced by `fetch_fixture_json` on a non-200 response — so `contract_validator.py` and the Step 4 tests exercise the exact error path — but explicitly labeled `_synthetic: true` with the reason embedded.

In [14]:
synthetic = {
    "_fixture_label": "error_500",
    "_view_name": "vlanlchangerequesttask",
    "_db_name": "dataportal",
    "_http_status": 500,
    "_fetched_at": fetched_at,
    "_source_env": SOURCE_ENV,
    "_synthetic": True,
    "_synthetic_reason": ("Pre-migration 12-view HTTP-500 cohort no longer reproduces on "
                          "post-migration dev (all 30 prod-origin vlanl*task views return 200, "
                          "2026-07-28 diagnostic). Retained synthetically per contract rev 10 b "
                          "to keep the fetch_status='error' code path under test."),
    "response": None,
    "_error_body": "Internal Server Error (synthetic placeholder -- see _synthetic_reason)",
}
path = os.path.join(OUT_DIR, "error_500.json")
with open(path, "w") as f:
    json.dump(synthetic, f, indent=2)
print("Wrote synthetic", path)

Wrote synthetic fixtures\error_500.json


## Redaction spot-check on `custom_tab.json`

In [15]:
import re
raw = open(os.path.join(OUT_DIR, "custom_tab.json")).read()
checks = {
    "raw emails remaining": len(re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", raw.replace("REDACTED_EMAIL", ""))),
    "pbplus URLs remaining": raw.count("pbplus.lanl.gov"),
    "user=<uid> remaining": len(re.findall(r"[?&;]user=(?!REDACTED_UID)[^&;\s\"]+", raw, re.I)),
    "REDACTED markers present": raw.count("REDACTED"),
    "customTabPropertyMap present": "customTabPropertyMap" in raw,
    "connectionUris present": "connectionUris" in raw,
}
for k, v in checks.items():
    print(f"  {k}: {v}")
print("\nExpect: 0 for the first three; markers > 0; both presence checks True.")
print("\nAll amendments done. Next: expected/<label>.py per template, then contract_validator.py.")

  raw emails remaining: 0
  pbplus URLs remaining: 0
  user=<uid> remaining: 0
  REDACTED markers present: 5
  customTabPropertyMap present: True
  connectionUris present: True

Expect: 0 for the first three; markers > 0; both presence checks True.

All amendments done. Next: expected/<label>.py per template, then contract_validator.py.
